# Consulta — Índices EEG adicionales para clasificación en BCI

## Introducción general

Las señales EEG basadas en Motor Imagery (MI) proporcionan interacción y comunicación entre pacientes con parálisis y el mundo exterior para mover y controlar dispositivos externos como sillas de ruedas y cursores. Sin embargo, los enfoques actuales en el diseño de sistemas MI-BCI requieren métodos efectivos de extracción de características y algoritmos de clasificación para adquirir características discriminativas de las señales EEG, debido a su naturaleza no lineal y no estacionaria[1].

Los métodos de extracción de características tradicionales en BCI se clasifican en dominio temporal, frecuencial, espacial y tiempo-frecuencial. Adicionalmente, métodos como el análisis de redes complejas y las características basadas en entropía ofrecen nuevas perspectivas para analizar las señales EEG que complementan la densidad espectral de potencia.

## Índice 1 — ERD/ERS (Event-Related Desynchronization / Synchronization)

### Descripción de la medida

La imaginación de un movimiento motor elicita una atenuación de corta duración de los ritmos en las bandas de frecuencia alpha (8-13 Hz) y beta (13-25 Hz), llamada desincronización relacionada con eventos (ERD). Esta ERD es similar a la actividad cortical durante la ejecución real del movimiento, lo que hace que el motor imagery sea adecuado para aplicaciones BCI en rehabilitación de pacientes con déficits motores[2].

El ERD se define como una disminución de la potencia del EEG relativa a un estado de reposo precedente dentro de las bandas mu (8-13 Hz) y beta (14-30 Hz), correlacionada con la ejecución motora, el motor imagery y la observación motora. La ERS es un rebote de la potencia en la banda beta después de la terminación del movimiento o la imaginación[3].

### Fórmula

$$ERD/ERS(\%) = \frac{P_{tarea} - P_{reposo}}{P_{reposo}} \times 100$$

Donde $P_{tarea}$ es la potencia durante la tarea y $P_{reposo}$ es la potencia durante el reposo (línea base). Valores negativos indican ERD (desincronización) y valores positivos indican ERS (sincronización).

### Relevancia para la clasificación en BCI

La ERD durante el motor imagery de muñeca se asocia con amplitudes de MEP significativamente aumentadas y SICI reducida, lo que demuestra que la magnitud de la ERD representa la excitabilidad de M1. Este estudio proporciona evidencia electrofisiológica de que una tarea de motor imagery que involucra ERD puede inducir cambios en la excitabilidad corticoespinal similares a los que acompañan a los movimientos reales[4].

Las BCI basadas en motor imagery explotan la modulación de los ritmos sensoriomotores sobre las cortezas motoras, conocidos como ERD y ERS. La interpretación del ERD/ERS está directamente relacionada con la selección de la línea base utilizada para estimarlos, lo cual puede resultar en una visualización engañosa si no se realiza correctamente[5].

### Librerías utilizadas

`scipy.signal` y `numpy` 

### Uso en los datos del proyecto

In [2]:
from scipy import signal
import numpy as np

def calcular_erd_ers(raw_segmento, raw_reposo, fs, banda):
    """
    Calcula ERD/ERS para un segmento de tarea respecto al reposo.
    
    Parámetros:
        raw_segmento: array (n_canales, n_muestras) — segmento T1 o T2
        raw_reposo:   array (n_canales, n_muestras) — segmento T0
        fs: frecuencia de muestreo (160 Hz)
        banda: tupla (f_min, f_max) — ej: (8, 13) para alpha
    
    Retorna:
        erd_ers: array (n_canales,) con el porcentaje ERD/ERS por canal
    """
    f_min, f_max = banda
    resultado = {}
    
    for idx in range(raw_segmento.shape[0]):
        # PSD de la tarea
        freqs, psd_tarea = signal.welch(
            raw_segmento[idx], fs=fs, nperseg=int(fs*2))
        # PSD del reposo (línea base)
        freqs, psd_reposo = signal.welch(
            raw_reposo[idx], fs=fs, nperseg=int(fs*2))
        
        # Índices de la banda de interés
        idx_banda = np.where((freqs >= f_min) & (freqs <= f_max))[0]
        
        # Potencia promedio en la banda
        p_tarea = np.mean(psd_tarea[idx_banda])
        p_reposo = np.mean(psd_reposo[idx_banda])
        
        # ERD/ERS como porcentaje de cambio relativo
        erd_ers = ((p_tarea - p_reposo) / (p_reposo + 1e-10)) * 100
        resultado[idx] = erd_ers
    
    return resultado

# En nuestros datos:
# Se espera ERD negativo en alpha y beta de C3 durante imaginación mano derecha (T2)
# Se espera ERD negativo en alpha y beta de C4 durante imaginación mano izquierda (T1)
# Esto refleja la dominancia contralateral del control motor

## Índice 2 — Parámetros de Hjorth

### Descripción de la medida

Los parámetros de Hjorth describen la señal EEG mediante tres medidas en el dominio temporal. La Actividad mide el cuadrado de la desviación estándar de la amplitud, es decir, la varianza de las señales EEG en una ventana de análisis. La Movilidad se define como la potencia media de la derivada normalizada de las señales EEG. La Complejidad representa la razón entre las movilidades del EEG, relacionando la primera derivada y la señal origina[6].

### Fórmulas

$$Actividad = \sigma^2(x)$$

$$Movilidad = \frac{\sigma(x')}{\sigma(x)}$$

$$Complejidad = \frac{Movilidad(x')}{Movilidad(x)}$$

Donde $x$ es la señal EEG, $x'$ es su primera derivada y $\sigma$ es la desviación estándar.

**Interpretación física:**
- **Actividad** → potencia total de la señal
- **Movilidad** → frecuencia media dominante
- **Complejidad** → qué tan similar es la señal a una sinusoide pura (complejidad = 1 para sinusoide perfecta)

### Relevancia para la clasificación en BCI

Al analizar las señales EEG con el parámetro de Hjorth en sistemas BCI, la precisión de clasificación mejoró en un 4.4% en promedio. Dado que las señales EEG tienen una propiedad no estacionaria y sus características de frecuencia difieren de individuo a individuo, los parámetros de Hjorth permiten seleccionar la banda de frecuencia y el tiempo dominantes para extraer características discriminantes[7].

Un promedio de precisión del 89.7 ± 0.78% fue obtenido al discriminar características de ejecución motora e imaginación motora en los canales C3, Cz y C4 usando parámetros de Hjorth combinados con FastICA y SVM, confirmando su efectividad para diferenciar estados motores en BCI[6]

### Librerías utilizadas

`numpy` 

### Uso en los datos del proyecto

In [3]:
import numpy as np

def calcular_hjorth(señal):
    """
    Calcula los 3 parámetros de Hjorth sobre un segmento EEG.
    
    Parámetros:
        señal: array 1D con el segmento de un canal EEG
    
    Retorna:
        actividad: varianza de la señal (potencia total)
        movilidad: frecuencia media de la señal
        complejidad: similitud con señal sinusoidal pura
    """
    # Actividad: varianza de la señal
    actividad = np.var(señal)
    
    # Primera derivada (diferencias entre muestras consecutivas)
    primera_derivada = np.diff(señal)
    
    # Movilidad: proporción de desviaciones estándar
    movilidad = np.std(primera_derivada) / (np.std(señal) + 1e-10)
    
    # Segunda derivada
    segunda_derivada = np.diff(primera_derivada)
    
    # Complejidad: cambio en la movilidad
    movilidad_deriv = np.std(segunda_derivada) / (np.std(primera_derivada) + 1e-10)
    complejidad = movilidad_deriv / (movilidad + 1e-10)
    
    return actividad, movilidad, complejidad

# En nuestros datos:
# Se aplica sobre segmentos T0 (reposo), T1 (mano izq), T2 (mano der)
# en los 9 canales de la grilla sensoriomotora
# Se espera que la movilidad y complejidad cambien durante motor imagery
# reflejando la reorganización de la actividad cortical

## Índice 3 — Entropía Espectral

### Descripción de la medida

La entropía de información, con sus características no lineales intrínsecas, captura efectivamente el comportamiento dinámico de las señales EEG, abordando las limitaciones de los métodos lineales tradicionales. Entre las 13 medidas de entropía más utilizadas en MI-BCI publicadas entre 2019 y 2023, la entropía de Shannon (13%) y la entropía espectral son de las más frecuentemente empleadas en extracción de características para motor imagery[7].

La entropía espectral mide qué tan uniforme o concentrada está la distribución de potencia en el espectro de frecuencias, basándose en la entropía de Shannon aplicada a la PSD normalizada:

$$H_{espectral} = -\sum_{f} p(f) \cdot \log_2(p(f))$$

Donde $p(f)$ es la PSD normalizada para que sume 1 (distribución de probabilidad):

$$p(f) = \frac{PSD(f)}{\sum_f PSD(f)}$$

**Interpretación física:**
- **Alta entropía** → potencia distribuida en muchas frecuencias → señal compleja y aleatoria
- **Baja entropía** → potencia concentrada en pocas frecuencias → señal más regular y ordenada

### Relevancia para la clasificación en BCI

Los métodos basados en entropía son los más adecuados para la caracterización de señales no estacionarias como el EEG. La entropía espectral combinada con correlación cruzada ha demostrado ser un método robusto de extracción de características para el control de dispositivos robóticos asistivos mediante motor imagery EEG[8].

Las características espectrales de energía, entropía y varianza de las sub-bandas de frecuencia del EEG (δ, θ, α, β y γ) calculadas mediante FFT son ampliamente utilizadas en sistemas BCI, extrayéndose de múltiples canales EEG para mejorar significativamente la clasificación de tareas de motor imagery.

### Librerías utilizadas

`scipy.signal` y `numpy`

### Uso en los datos del proyecto

In [4]:
from scipy import signal
import numpy as np

def calcular_entropia_espectral(señal, fs, f_min, f_max):
    """
    Calcula la entropía espectral de una banda de frecuencia.
    
    Parámetros:
        señal: array 1D con el segmento de un canal EEG
        fs: frecuencia de muestreo (160 Hz en nuestros datos)
        f_min, f_max: límites de la banda (ej: 8, 13 para alpha)
    
    Retorna:
        entropia: valor escalar de entropía espectral en la banda
    """
    # Calcular PSD con Welch (mismo método del proyecto 1)
    freqs, psd = signal.welch(señal, fs=fs, nperseg=int(fs*2))
    
    # Filtrar frecuencias de la banda de interés
    idx_banda = np.where((freqs >= f_min) & (freqs <= f_max))[0]
    psd_banda = psd[idx_banda]
    
    # Normalizar para obtener distribución de probabilidad
    psd_norm = psd_banda / (np.sum(psd_banda) + 1e-10)
    
    # Entropía de Shannon sobre el espectro normalizado
    entropia = -np.sum(psd_norm * np.log2(psd_norm + 1e-10))
    
    return entropia

# En nuestros datos:
# Se calcula por banda (alpha y beta) en los 9 canales sensoriomotores
# para segmentos T0, T1 y T2
# Se espera mayor entropía durante motor imagery que en reposo
# indicando mayor complejidad de la actividad cortical durante imaginación

# Plan de Análisis — Segundo Proyecto BCI

## 1. Contexto y condiciones de estudio

En el proyecto 1 se procesaron 109 sujetos con 1526 archivos EDF, aplicando un pipeline de filtrado (pasa banda 0.5-40 Hz + Notch 60 Hz) sobre una grilla sensoriomotora de 9 canales (Fc3, Fcz, Fc4, C3, Cz, C4, Cp3, Cpz, Cp4)[6][3] y se calculó la PSD por bandas usando el método de Welch.

Para este proyecto, el análisis se enfoca en 3 condiciones específicas extraídas de los segmentos T0, T1 y T2 de los runs de imaginación (R04, R08, R12):

| Condición | Etiqueta | Runs | Descripción |
|---|---|---|---|
| Reposo | T0 | R04, R08, R12 | Sin actividad motora — línea base |
| Imaginación mano derecha | T2 | R04, R08, R12 | Motor imagery mano derecha |
| Imaginación mano izquierda | T1 | R04, R08, R12 | Motor imagery mano izquierda |



## 2. Pipeline de procesamiento (heredado del proyecto 1)

Los 9 canales de la grilla sensoriomotora (Fc3, Fcz, Fc4, C3, Cz, C4, Cp3, Cpz, Cp4) se justifican con base en los resultados del proyecto 1, donde se obtuvo una precisión promedio del 89.7 ± 0.78% al discriminar características de ejecución motora e imaginación motora en los canales C3, Cz y C4, confirmando que estos canales son los más relevantes para motor imagery.Las señales EEG son preprocesadas usando filtros FIR, seguidos de extracción de características multidominio incluyendo energía, patrones espaciales comunes y densidad espectral de potencia, que son fusionadas para mejorar la precisión de clasificación[10].

El procesamiento ya validado en el proyecto 1 se mantiene:

```
Señal EDF cruda (64 canales, 160 Hz, ~125s por archivo)
    ↓
Selección grilla sensoriomotora 3×3
(Fc3, Fcz, Fc4, C3, Cz, C4, Cp3, Cpz, Cp4)
    ↓
Filtro pasa banda FIR (0.5 – 40 Hz)
    ↓
Filtro Notch (60 Hz)
    ↓
Segmentación por anotaciones
T0 = reposo (~4.2s) | T1 = mano izq (~4.1s) | T2 = mano der (~4.1s)
    ↓
Extracción de características por segmento y canal
```



## 3. Extracción de características


Para cada segmento y canal se extraen 4 grupos de características:

### 3.1 PSD por bandas (proyecto 1 — ya calculada)

La PSD calculada con el método de Welch en el proyecto 1 es la base del análisis espectral. Un enfoque bien establecido en BCI basados en motor imagery ha sido usar la PSD de las señales como características para discriminar entre tareas motoras, obteniendo precisiones medias del 90 ± 8% y 87 ± 7% para las bandas mu y beta respectivamente[6].

Potencia promedio en cada banda usando método de Welch con ventana de 2 segundos (resolución espectral 0.5 Hz):

$$PSD_{banda} = \frac{1}{|B|} \sum_{f \in B} S(f)$$

Donde $S(f)$ es la densidad espectral y $B$ es el conjunto de frecuencias de la banda.

**Resultado:** 5 valores por canal → 45 características totales (5 bandas × 9 canales)

### 3.2 ERD/ERS (nuevo índice 1)

La imaginación de un movimiento motor elicita una atenuación de corta duración de los ritmos en las bandas alpha (8-13 Hz) y beta (13-25 Hz), llamada ERD. Esta ERD es similar a la actividad cortical durante la ejecución real del movimiento, lo que hace que el motor imagery sea adecuado para sistemas BCI en rehabilitación[2].

Cuantifica el cambio relativo de potencia durante la imaginación respecto al reposo:

$$ERD/ERS(\%) = \frac{P_{tarea} - P_{reposo}}{P_{reposo}} \times 100$$


**Resultado:** 2 valores por canal → 18 características (2 bandas × 9 canales)

### 3.3 Parámetros de Hjorth (nuevo índice 2)

Al analizar las señales EEG con el parámetro de Hjorth en sistemas BCI, la precisión de clasificación mejora un 4.4% en promedio. Los parámetros permiten seleccionar la banda de frecuencia y el tiempo dominantes para extraer características discriminantes, siendo especialmente útiles dado que las señales EEG tienen propiedades no estacionarias que varían entre individuos[7].

Describen la señal en el dominio temporal:

$$Actividad = \sigma^2(x) \qquad Movilidad = \frac{\sigma(x')}{\sigma(x)} \qquad Complejidad = \frac{Movilidad(x')}{Movilidad(x)}$$

**Resultado:** 3 valores por canal → 27 características (3 parámétricas × 9 canales)

### 3.4 Entropía Espectral (nuevo índice 3)

La entropía de información, con sus características no lineales intrínsecas, captura efectivamente el comportamiento dinámico de las señales EEG, abordando las limitaciones de los métodos lineales tradicionales en la captura de características no lineales. Entre las medidas más utilizadas en MI-BCI entre 2019 y 2023, la entropía de Shannon es de las más frecuentemente empleadas[8].

Mide la complejidad de la distribución espectral:

$$H = -\sum_{f} p(f) \cdot \log_2(p(f))$$

Se calcula por banda de frecuencia.

**Resultado:** 5 valores por canal → 45 características (5 bandas × 9 canales)



## 4. Estructura del DataFrame final

| Columna | Tipo | Ejemplo |
|---|---|---|
| sujeto | str | S001 |
| condicion | str | imaginacion_mano_der |
| psd_alpha_C3 | float | 38.5 µV²/Hz |
| erd_alpha_C3 | float | -23.4% |
| erd_beta_C3 | float | -18.7% |
| hjorth_actividad_C3 | float | 0.0023 |
| hjorth_movilidad_C3 | float | 0.42 |
| hjorth_complejidad_C3 | float | 1.87 |
| entropia_alpha_C3 | float | 3.21 |
| ... | ... | ... |

**Total de características:** ~135 características × 9 canales


## 5. Análisis estadístico

### 5.1 Estadística descriptiva

Se usarán las representaciones recomendadas por data-to-viz.com para comparar las 3 condiciones:

- **Boxplots** → comparar distribuciones de ERD/ERS entre condiciones en C3 y C4
- **Violin plots** → mostrar forma de distribución de Hjorth y entropía
- **Heatmap** → ERD/ERS por canal y condición — permite ver lateralización
- **Barplot con error** → media ± desviación estándar por condición

### 5.2 Prueba de normalidad

Shapiro-Wilk sobre cada índice y condición (máximo 50 muestras):

- $H_0$: Los datos siguen distribución normal
- $H_1$: Los datos **NO** siguen distribución normal

Basado en los resultados del proyecto 1, se espera que los datos no sean normales → pruebas no paramétricas.

### 5.3 Hipótesis

**Hipótesis 1 — ERD en alpha y beta durante motor imagery:**

- $H_0$: No hay diferencia en ERD entre reposo e imaginación de movimiento
- $H_1$: El ERD es significativamente más negativo durante imaginación que en reposo
- **Prueba:** Mann-Whitney U (reposo vs imaginación mano der, reposo vs imaginación mano izq)

**Hipótesis 2 — Lateralización hemisférica C3 vs C4:**

- $H_0$: No hay diferencia entre C3 y C4 durante imaginación de mano derecha vs izquierda
- $H_1$: La imaginación de mano derecha produce mayor ERD en C3 y la de mano izquierda en C4
- **Prueba:** Mann-Whitney U (C3 vs C4 por condición)

**Hipótesis 3 — Diferenciación entre manos:**

- $H_0$: Los índices no permiten discriminar entre imaginación de mano derecha e izquierda
- $H_1$: Existe diferencia significativa en al menos un índice entre imaginación mano derecha e izquierda
- **Prueba:** Mann-Whitney U (T1 vs T2 en runs de imaginación)


## 6. Clasificación

### 6.1 Selección de características

Se seleccionan los índices con mayor diferencia estadística entre condiciones, usando como criterio:

- p-valor < 0.05 en pruebas de hipótesis
- Mayor separabilidad visual en boxplots

### 6.2 Estrategia de entrenamiento — K-Fold Cross Validation

Los enfoques actuales en el diseño de sistemas MI-BCI requieren métodos efectivos de extracción de características y algoritmos de clasificación para adquirir características discriminativas, debido a la naturaleza no lineal y no estacionaria de las señales EEG. El K-Fold garantiza una evaluación robusta e imparcial del clasificador[1].

Se usa K-Fold con k=5 para evaluar el rendimiento de los clasificadores de forma robusta:

```
Dataset completo (109 sujetos)
         ↓
    K-Fold (k=5)
    ┌─────────────────────────────┐
    │ Fold 1: 80% train, 20% test │
    │ Fold 2: 80% train, 20% test │
    │ Fold 3: 80% train, 20% test │
    │ Fold 4: 80% train, 20% test │
    │ Fold 5: 80% train, 20% test │
    └─────────────────────────────┘
         ↓
  Promedio de métricas
  (accuracy, F1-score, AUC-ROC)
```

### 6.3 Arquitecturas de clasificación [11]

| Modelo | Librería | Por qué usarlo |
|---|---|---|
| MLP (Red neuronal) | sklearn / keras | Modelo base de referencia |
| CNN | keras/tensorflow | Captura patrones espaciales en la grilla 3×3 |
| LSTM | keras/tensorflow | Captura dependencias temporales en EEG |
| SVM | sklearn | Robusto con pocas muestras, estándar en BCI |
| XGBoost | xgboost | Alto rendimiento en datos tabulares |

### 6.4 Métricas de evaluación

- **Accuracy** → porcentaje de clasificaciones correctas
- **F1-score** → balance entre precisión y recall
- **Matriz de confusión** → errores por clase
- **AUC-ROC** → capacidad discriminante del modelo

## Referencias

[1]Oh, S.-H., Lee, Y.-R., & Kim, H.-N. (2014). A novel EEG feature extraction method using Hjorth parameter. International Journal of Electronics and Electrical Engineering, 2(2), 106–110. https://doi.org/10.12720/ijeee.2.2.106-110

[2]Zhang, S., Wang, S., Zheng, D., Zhu, K., & Dai, M. (2019). A novel pattern with high-level commands for encoding motor imagery-based brain computer interface. Pattern Recognition Letters, 125, 574–580. https://doi.org/10.1016/j.patrec.2019.03.020

[3]Lotte, F., Van Langhenhove, A., Lamarche, F., Ernest, T., Renard, Y., Arnaldi, B., & Lécuyer, A. (2010). Exploring large virtual environments by thoughts using a brain–computer interface based on motor imagery and high-level commands. Presence: Teleoperators and Virtual Environments, 19(1), 54–70. https://doi.org/10.1162/pres.19.1.54

[4]Bodda, S., & Diwakar, S. (2022). Exploring EEG spectral and temporal dynamics underlying a hand grasp movement. PLOS ONE, 17(6), e0270366. https://doi.org/10.1371/journal.pone.0270366

[5]Kauati-Saito, E., Pereira, A. da S., Fontana, A. P., de Sá, A. M. F. L. M., Soares, J. G. M., & Tierra-Criollo, C. J. (2025). Classification of different motor imagery tasks with the same limb using electroencephalographic signals. Sensors, 25(17), 5291. https://doi.org/10.3390/s25175291

[6]Wang, Y., Gao, X., Hong, B., & Gao, S. (2010). Practical designs of brain-computer interfaces based on the modulation of EEG rhythms. En B. Graimann, G. Pfurtscheller, & B. Allison (Eds.), Brain-computer interfaces: Revolutionizing human-computer interaction (pp. 137–154). Springer. https://doi.org/10.1007/978-3-642-02091-9_8

[7]Degirmenci, M., Yuce, Y. K., Perc, M., & Isler, Y. (2024). EEG channel and feature investigation in binary and multiple motor imagery task predictions. Frontiers in Human Neuroscience, 18, 1525139. https://doi.org/10.3389/fnhum.2024.1525139

[8]Kitahara, K., Hayashi, Y., Yano, S., & Kondo, T. (2017). Target-directed motor imagery of the lower limb enhances event-related desynchronization. PLOS ONE, 12(9), e0184245. https://doi.org/10.1371/journal.pone.0184245

[9]Yakovlev, L., Syrov, N., Miroshnikov, A., Lebedev, M., & Kaplan, A. (2023). Event-related desynchronization induced by tactile imagery: an EEG study. eNeuro, 10(6), ENEURO.0455-22.2023. https://doi.org/10.1523/ENEURO.0455-22.2023

[10]Prasad, A. (2016). Feature extraction and classification for motor imagery in EEG signals [Proyecto final de Maestría, Kaunas University of Technology]. Kaunas University of Technology.

[11]Kaushik, G., Gaur, P., Sharma, R. R., & Pachori, R. B. (2022). EEG signal based seizure detection focused on Hjorth parameters from tunable-Q wavelet sub-bands. Biomedical Signal Processing and Control, 76, 103645. https://doi.org/10.1016/j.bspc.2022.103645